In [ ]:
import logging
import os
import tempfile
import warnings

# This tutorial does not need a GPU. Keep Molab and headless documentation
# builds away from host CUDA plugins before importing HSSM/JAX.
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_SKIP_CUDA_CONSTRAINTS_CHECK"] = "1"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
warnings.filterwarnings("ignore")
logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)

# Work in a throwaway folder so the tutorial leaves nothing behind, and
# ignore any tracking server configured in the environment.
os.environ.pop("MLFLOW_TRACKING_URI", None)
os.chdir(tempfile.mkdtemp(prefix="hssm-mlflow-"))

import arviz as az
import marimo as mo
import mlflow
import pandas as pd

import hssm

# MLflow and PyMC set their log levels on import, so quiet them afterwards.
for _name in ("alembic", "mlflow", "pymc"):
    logging.getLogger(_name).setLevel(logging.WARNING)

FULL_RUN = os.environ.get("FULL_RUN") == "1"
SAMPLER = {
    "draws": 1_000 if FULL_RUN else 100,
    "tune": 1_000 if FULL_RUN else 100,
    "chains": 2,
    "cores": 1,
    "random_seed": 123,
    "progressbar": False,
}

# Track fits with MLflow

`hssm.track` records each fit as an [MLflow](https://mlflow.org) run. This
notebook shows the three things that gives you:

1. **Tracked runs**: the model, its settings and diagnostics, recorded
   without extra code, and easy to find again;
2. **Kept artifacts**: the traces, the model specification and the data
   the model was fit to, stored with the run;
3. **Rebuilt models**: a run turned back into a fitted model, in any
   session.

Tracking is off by default and needs the optional extra:
`uv add "hssm[tracking]"` or `pip install "hssm[tracking]"`.

The setup cell above moves into a throwaway folder first. With nothing
else configured, MLflow keeps its records in `mlflow.db` and the run files
in `mlruns/` in your working folder; here that folder is discarded
afterwards. The [how-to](../how_to/track_with_mlflow.md) covers using a
shared store instead.

## Record a fit

Everything inside the `hssm.track` block becomes one run. Naming the
dataset is optional; it fills the run's **Datasets** panel in the MLflow UI.

In [ ]:
data = hssm.load_data("cavanagh_theta")

with hssm.track(
    experiment="my-study",
    run_name="ddm-basic",
    dataset_name="cavanagh_theta",
) as run:
    model = hssm.HSSM(data, model="ddm")
    model.sample(**SAMPLER)

first_run_id = run.run_id
mo.md(f"Recorded run `{first_run_id}`.")

MLflow tracking: started run 8375c159635a41bcbdc166fa7a9a5a6c
Model initialized successfully.
Using default initvals. 



Recorded run `8375c159635a41bcbdc166fa7a9a5a6c`.

## What a run holds

The run records the model and its settings as **params**, the fit's
diagnostics as **metrics**, and identifying labels as **tags**. Two tags
group runs later: `spec_sha256` (a param) changes with the modelling
choices, and `data_sha256` with the data.

In [ ]:
_run = mlflow.get_run(first_run_id)
_params = pd.DataFrame(sorted(_run.data.params.items()), columns=["param", "value"])
_metrics = pd.DataFrame(
    sorted((k, round(v, 3)) for k, v in _run.data.metrics.items()),
    columns=["metric", "value"],
)
mo.md(
    "### Params\n\n"
    + _params.to_markdown(index=False)
    + "\n\n### Metrics\n\n"
    + _metrics.to_markdown(index=False)
)

### Params

| param         | value                                                            |
|:--------------|:-----------------------------------------------------------------|
| bambi_version | 0.19.0                                                           |
| chains        | 2                                                                |
| cores         | 1                                                                |
| draws         | 1000                                                             |
| hssm_version  | 0.5.0                                                            |
| loglik_kind   | analytical                                                       |
| model         | ddm                                                              |
| n_subjects    | 14                                                               |
| n_trials      | 3988                                                             |
| pymc_version  | 6.1.0                                                            |
| sampler       | pymc                                                             |
| spec_sha256   | 5dc628101d7ab6726dd1dea594552ed76d95b07f853ff4fe108e2568532a9a6a |
| tune          | 1000                                                             |

### Metrics

| metric           |    value |
|:-----------------|---------:|
| divergences      |    0     |
| ess_bulk_min     | 1210     |
| ess_tail_min     |  975     |
| r_hat_max        |    1     |
| sampling_seconds |  124.902 |

The run also keeps the files needed to reuse it. `data.parquet` is the
data the model was fit to, covariates included; the traces keep only the
observed response.

In [ ]:
_files = mlflow.MlflowClient().list_artifacts(first_run_id)
_kept = pd.DataFrame(
    [(f.path, f"{f.file_size / 1024:,.1f} KB") for f in _files],
    columns=["file", "size"],
)
mo.md(_kept.to_markdown(index=False))

| file            | size        |
|:----------------|:------------|
| data.parquet    | 51.2 KB     |
| model_spec.json | 0.9 KB      |
| summary.csv     | 0.3 KB      |
| traces.nc       | 62,813.4 KB |

The files sit under the working folder, in
`mlruns/<experiment id>/<run id>/artifacts/`. The run reports that path as
its `artifact_uri`, and `mlflow.artifacts.download_artifacts` fetches a
file by name:

In [ ]:
_uri = mlflow.get_run(first_run_id).info.artifact_uri
_path = mlflow.artifacts.download_artifacts(
    run_id=first_run_id, artifact_path="data.parquet"
)
_stored = pd.read_parquet(_path)
mo.md(
    f"Files of this run, relative to the working folder: `{os.path.relpath(_uri)}`\n\n"
    f"Its `data.parquet` holds {len(_stored):,} rows with the columns "
    + ", ".join(f"`{c}`" for c in _stored.columns)
    + "."
)

Files of this run, relative to the working folder: `mlruns/1/8375c159635a41bcbdc166fa7a9a5a6c/artifacts`

Its `data.parquet` holds 3,988 rows with the columns `participant_id`, `stim`, `rt`, `response`, `theta`, `dbs`, `conf`.

## Vary something and compare

`params=` records what you are varying, and `run.log_metric` adds your own
numbers. Here two prior widths for the drift rate `v` are fit in separate
runs, one block per fit, each logging its posterior mean of `v`.

In [ ]:
for _sd in [0.5, 2.0]:
    with hssm.track(
        experiment="my-study",
        run_name=f"ddm-v-sd-{_sd}",
        params={"prior_v_sd": _sd},
        dataset_name="cavanagh_theta",
    ) as _run:
        _model = hssm.HSSM(
            data,
            model="ddm",
            v={"prior": {"name": "Normal", "mu": 0.0, "sigma": _sd}},
        )
        _model.sample(**SAMPLER)
        _run.log_metric("v_mean", float(_model.traces.posterior["v"].mean()))

MLflow tracking: started run 26e9bcdfe7ad4c6698a63b193e3fa6e4
Model initialized successfully.
Using default initvals. 

MLflow tracking: started run 07d1aa9ed9c348529152eb77cab1ebeb
Model initialized successfully.
Using default initvals. 



MLflow's own `search_runs` filters on anything a run recorded, including
your params and metrics:

In [ ]:
_compared = mlflow.search_runs(
    experiment_names=["my-study"],
    filter_string="params.prior_v_sd != ''",
    order_by=["params.prior_v_sd"],
)[
    [
        "tags.mlflow.runName",
        "params.prior_v_sd",
        "metrics.v_mean",
        "metrics.r_hat_max",
    ]
]
mo.md(_compared.round(3).to_markdown(index=False))

| tags.mlflow.runName   |   params.prior_v_sd |   metrics.v_mean |   metrics.r_hat_max |
|:----------------------|--------------------:|-----------------:|--------------------:|
| ddm-v-sd-0.5          |                 0.5 |            0.389 |                   1 |
| ddm-v-sd-2.0          |                 2   |            0.391 |                   1 |

## Find runs again

`hssm.list_runs()` lists every run `hssm.track` recorded, newest first,
one row per run. Its `restorable` column says whether the run can be
rebuilt. It takes no filters; filter the DataFrame instead.

In [ ]:
runs = hssm.list_runs()
_ddm = runs[runs["model"] == "ddm"]
mo.md(
    _ddm[
        [
            "run_name",
            "experiment",
            "model",
            "dataset_name",
            "n_trials",
            "restorable",
        ]
    ].to_markdown(index=False)
)

| run_name     | experiment   | model   | dataset_name   |   n_trials | restorable   |
|:-------------|:-------------|:--------|:---------------|-----------:|:-------------|
| ddm-v-sd-2.0 | my-study     | ddm     | cavanagh_theta |       3988 | True         |
| ddm-v-sd-0.5 | my-study     | ddm     | cavanagh_theta |       3988 | True         |
| ddm-basic    | my-study     | ddm     | cavanagh_theta |       3988 | True         |

## Rebuild a model from a run

`hssm.load_run` rebuilds the model from the run's `model_spec.json` and
`data.parquet`, and attaches its traces. No pickle is involved, and a
likelihood network from Hugging Face is fetched at the revision the fit
used.

In [ ]:
rebuilt = hssm.load_run(first_run_id)
mo.md(az.summary(rebuilt.traces).round(3).to_markdown())

Model initialized successfully.


|    |   mean |     sd |   eti89_lb |   eti89_ub |   ess_bulk |   ess_tail |   r_hat |   mcse_mean |   mcse_sd |
|:---|-------:|-------:|-----------:|-----------:|-----------:|-----------:|--------:|------------:|----------:|
| a  | 1.0641 | 0.0084 |       1.1  |       1.1  |       1360 |       1312 |       1 |     0.00023 |   0.00015 |
| t  | 0.348  | 0.0052 |       0.34 |       0.36 |       1303 |        975 |       1 |     0.00014 |   0.0001  |
| z  | 0.4959 | 0.0061 |       0.49 |       0.51 |       1210 |       1354 |       1 |     0.00018 |   0.00013 |
| v  | 0.391  | 0.0202 |       0.36 |       0.42 |       1271 |       1271 |       1 |     0.00057 |   0.0004  |

In [ ]:
_same = (
    rebuilt.traces["posterior"]
    .to_dataset()
    .equals(model.traces["posterior"].to_dataset())
)
mo.md(f"The rebuilt model's posterior matches the original fit: **{_same}**.")

The rebuilt model's posterior matches the original fit: **True**.

## Browse the runs

From the folder that holds `mlflow.db`, start the MLflow UI and open the
address it prints:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

The [Track fits with MLflow](../how_to/track_with_mlflow.md) how-to lists
everything a run records, how to attach your own files, and how to read a
store other lab members log to.